In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MultiLabelBinarizer, StandardScaler
from sklearn.model_selection import cross_val_score, KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')
 

c:\Users\asiae\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
c:\Users\asiae\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


In [7]:
import pandas as pd

files = {
    "cleanser":  r"C:\workspace\finalproject\data\model_data_v1\model_cleanser_final.csv",
    "skincare":  r"C:\workspace\finalproject\data\model_data_v1\model_skincare_final.csv",
    "maskpack":  r"C:\workspace\finalproject\data\model_data_v1\model_mask_final.csv",
    "suncare":   r"C:\workspace\finalproject\data\model_data_v1\model_suncare_final.csv",
}

cols = ["skin_type", "material_type_free", "product_benefits"]

for col in cols:
    all_values = set()
    for cat, path in files.items():
        df = pd.read_csv(path)
        if col not in df.columns:
            continue
        series = df[col].dropna().astype(str)
        series = series.str.replace(r'[-\s]+', '_', regex=True).str.lower()
        for row in series:
            for val in row.split(','):
                all_values.add(val.strip())
    
    print(f"\n[{col}] 유니크값 총 {len(all_values)}개")
    for i, v in enumerate(sorted(all_values), 1):
        print(f"  {i:>3}. {v}")


[skin_type] 유니크값 총 159개
    1. _5_five_star_award_winning_vitamin_c_seurm_lily_sado
    2. _acne
    3. _acne_prone
    4. _acne_prone)
    5. _acne_prone_skin
    6. _acneic
    7. _adults
    8. _ageing
    9. _aging
   10. _aging_skin
   11. _all
   12. _all_skin_type
   13. _all_skin_types
   14. _ance_prone
   15. _and_ages
   16. _and_oily_skin
   17. _anti_aging
   18. _anti_aging_anti_wrinkle_vegan_organic_naturalfor_all_skin_types_normal_comination_oily
   19. _anti_aging_anti_wrinkle_v…_see_more
   20. _balanced
   21. _blemish
   22. _blemish_skin
   23. _blemished
   24. _blemished_and/or_acne_prone.
   25. _break_out
   26. _bumpy
   27. _combi
   28. _combination
   29. _combination_skin
   30. _cracking
   31. _damaged
   32. _delicate
   33. _dhc_deep_cleansing_oil
   34. _dry
   35. _dry_and_sensitive
   36. _dry_combination
   37. _dry_or_mature_skin
   38. _dry_sensitive_acne_prone_blemishes_pore_minimizer
   39. _dry_skin
   40. _dull
   41. _enviromantally_damaged

In [11]:
import pandas as pd
import re

# ─────────────────────────────────────────
# skin_type 매핑
# ─────────────────────────────────────────
skin_type_map = {
    "oily":        ["oily", "oil_skin", "oil skin", "shiny", "sebum", "oil_production", "oil production"],
    "dry":         ["dry", "drier", "dehydrat", "cracking", "flaking"],
    "combination": ["combination", "combi", "combin", "combo"],
    "sensitive":   ["sensitiv", "sensetive", "sensitve", "sesitive", "irritat", "rosacea", "eczema", "fragile", "delicate"],
    "normal":      ["normal"],
    "acne_prone":  ["acne", "blemish", "breakout", "pimple", "zit", "ance", "problem"],
    "mature":      ["mature", "aging", "ageing", "wrinkle", "anti_aging", "anti aging", "firming"],
    # "all" 단독값 + 기존 패턴 모두 잡기
    "all":         ["all skin", "all_skin", "all type", "all_type", "universal", "any", "multi",
                    "for all", "every", "all condition", "all_condition", "^all$"],
}

# ─────────────────────────────────────────
# material_type_free 매핑
# ─────────────────────────────────────────
material_map = {
    "paraben_free":    ["paraben", "paraban", "parabeen", "praben"],
    "fragrance_free":  ["fragrance", "fragance", "scent", "unscented", "perfume"],
    "sulfate_free":    ["sulfate", "sulphate", "sls", "sles", "sodium_lauryl", "sodium lauryl"],
    "alcohol_free":    ["alcohol", "ethanol"],
    "cruelty_free":    ["cruelty", "animal_test", "animal test", "not_tested", "not tested", "no_animal", "no animal"],
    "vegan":           ["vegan"],
    "oil_free":        ["oil_free", "oil free"],
    "silicone_free":   ["silicone", "silicon"],
    "phthalate_free":  ["phthalate", "phthalat"],
    "gluten_free":     ["gluten"],
    "dye_free":        ["dye_free", "dye free", "colorant_free", "colorant free", "color_free"],
    "mineral_oil_free":["mineral_oil", "mineral oil"],
    "gmo_free":        ["gmo", "non_gmo", "non gmo"],
    "preservative_free":["preservative"],
    "triclosan_free":  ["triclosan"],
}

# ─────────────────────────────────────────
# product_benefits 매핑
# ─────────────────────────────────────────
benefit_map = {
    "moisturizing":    ["moistur", "hydrat", "nourish", "replenish", "water", "dehydrat"],
    "anti_aging":      ["anti_ag", "anti ag", "antiag", "wrinkle", "fine_line", "fine line", "aging", "ageing", "age_defying", "age defying"],
    "brightening":     ["brighten", "whitening", "glow", "radiant", "luminous", "lighten", "illuminat", "dark_spot", "dark spot", "pigment", "even_skin", "even skin", "even_ton", "even ton"],
    "soothing":        ["sooth", "calm", "redness", "irritat", "rosacea", "sensitiv", "anti_inflamm", "anti inflamm"],
    "exfoliating":     ["exfoliat", "peel", "resurface", "cell_renew", "cell renew"],
    "acne_control":    ["acne", "blemish", "breakout", "pimple", "zit", "blackhead", "whitehead", "pore_cleans", "pore cleans"],
    "pore_control":    ["pore_minim", "pore minim", "pore_tighten", "pore tighten", "pore_refin", "pore refin", "shrink_pore", "shrink pore", "minimize_pore", "minimize pore"],
    "sun_protection":  ["spf", "uv", "sunscreen", "sun_protect", "sun protect", "broad_spectrum", "broad spectrum"],
    "firming":         ["firm", "tighten", "lifting", "lift", "elasticity", "sagging", "plump"],
    "cleansing":       ["cleans", "detox", "purif", "remov", "dirt"],
    "oil_control":     ["oil_control", "oil control", "oil_free", "oil free", "mattif", "shine_control", "shine control"],
    "anti_oxidant":    ["antioxidant", "free_radical", "free radical"],
    "repair":          ["repair", "restor", "rebuild", "recover", "heal"],
    "ph_balance":      ["ph_balanc", "ph balanc", "ph_level", "ph level"],
}

# ─────────────────────────────────────────
# 정제 함수
# ─────────────────────────────────────────
def map_column(text, mapping):
    if pd.isna(text):
        return []
    text = str(text).strip().lower()
    text = re.sub(r'[-\s]+', '_', text)
    
    # "all" 단독값 처리
    if text == "all":
        return ["all"]
    
    matched = []
    for label, keywords in mapping.items():
        if any(kw in text for kw in keywords):
            matched.append(label)
    return matched if matched else ["other"]
# ─────────────────────────────────────────
# 전체 파일에 적용
# ─────────────────────────────────────────
files = {
    "cleanser": r"C:\workspace\finalproject\data\model_data_v1\model_cleanser_final.csv",
    "skincare": r"C:\workspace\finalproject\data\model_data_v1\model_skincare_final.csv",
    "mask":     r"C:\workspace\finalproject\data\model_data_v1\model_mask_final.csv",
    "suncare":  r"C:\workspace\finalproject\data\model_data_v1\model_suncare_final.csv",
}

for cat, path in files.items():
    df = pd.read_csv(path)

    df["skin_type_clean"]    = df["skin_type"].apply(lambda x: map_column(x, skin_type_map))
    df["material_clean"]     = df["material_type_free"].apply(lambda x: map_column(x, material_map))
    df["benefits_clean"]     = df["product_benefits"].apply(lambda x: map_column(x, benefit_map))

    out_path = path.replace("_final.csv", "_cleaned.csv")
    df.to_csv(out_path, index=False)
    print(f"[{cat}] 저장 완료 → {out_path}")

    # 정제 결과 샘플 확인
    print(df[["skin_type", "skin_type_clean", "product_benefits", "benefits_clean"]].head(3).to_string())
    print()


[cleanser] 저장 완료 → C:\workspace\finalproject\data\model_data_v1\model_cleanser_cleaned.csv
                             skin_type                   skin_type_clean product_benefits  benefits_clean
0  All, Oily, Combination, Dry, Normal  [oily, dry, combination, normal]      Exfoliating   [exfoliating]
1                          Combination                     [combination]     Moisturizing  [moisturizing]
2               Sensitive, Dry, Normal          [dry, sensitive, normal]      Exfoliating   [exfoliating]

[skincare] 저장 완료 → C:\workspace\finalproject\data\model_data_v1\model_skincare_cleaned.csv
                           skin_type                 skin_type_clean          product_benefits              benefits_clean
0                        Combination                   [combination]                 Hydration              [moisturizing]
1  Sensitive, Dry, Oily, Normal, All  [oily, dry, sensitive, normal]  Moisturizing, Anti-Aging  [moisturizing, anti_aging]
2                       

In [1]:
import pandas as pd
import ast

df = pd.read_csv(r"C:\workspace\finalproject\data\model_data_v1\model_cleanser_cleaned.csv")

# 컬럼 전체 확인 (y 컬럼명도 여기서 확인)
print(df.columns.tolist())

c:\Users\asiae\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
c:\Users\asiae\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


['parent_asin', 'title', 'item_form', 'skin_type', 'material_type_free', 'product_benefits', 'average_rating', 'rating_number', 'y2', 'model', 'y1', 'y_final', 'title_clean', 'item_form_norm', 'skin_type_norm', 'material_norm', 'benefit_norm', 'skin_type_clean', 'material_clean', 'benefits_clean']


In [2]:
print(df[["item_form", "item_form_norm", "skin_type", "skin_type_norm"]].head(5).to_string())

  item_form item_form_norm                            skin_type                                   skin_type_norm
0     Cream      ['cream']  All, Oily, Combination, Dry, Normal  ['normal', 'combination', 'dry', 'oily', 'all']
1     Cream      ['cream']                          Combination                                  ['combination']
2       Gel        ['gel']               Sensitive, Dry, Normal                   ['normal', 'sensitive', 'dry']
3     Cream      ['cream']                       Sensitive, Dry                             ['sensitive', 'dry']
4     Cream      ['cream']                            Sensitive                                    ['sensitive']


In [3]:
print(df[["material_type_free", "material_norm", "product_benefits", "benefit_norm"]].head(5).to_string())

                                              material_type_free                                                              material_norm          product_benefits                 benefit_norm
0  Paraben Free, Vegan, Cruelty Free, Sulfate Free, Alcohol Free  ['cruelty_free', 'alcohol_free', 'sulfate_free', 'vegan', 'paraben_free']               Exfoliating              ['exfoliating']
1                                                   Cruelty Free                                                           ['cruelty_free']              Moisturizing                ['hydrating']
2                                          Fragrance Free, Vegan                                                ['fragrance_free', 'vegan']               Exfoliating              ['exfoliating']
3                                            Vegan, Cruelty Free                                                  ['cruelty_free', 'vegan']              Moisturizing                ['hydrating']
4                        

In [4]:
import pandas as pd
import ast
from sklearn.preprocessing import MultiLabelBinarizer

files = {
    "cleanser": r"C:\workspace\finalproject\data\model_data_v1\model_cleanser_cleaned.csv",
    "skincare": r"C:\workspace\finalproject\data\model_data_v1\model_skincare_cleaned.csv",
    "mask":     r"C:\workspace\finalproject\data\model_data_v1\model_mask_cleaned.csv",
    "suncare":  r"C:\workspace\finalproject\data\model_data_v1\model_suncare_cleaned.csv",
}

list_cols = ["item_form_norm", "skin_type_norm", "material_norm", "benefit_norm"]

for cat, path in files.items():
    df = pd.read_csv(path)
    
    # 리스트 컬럼 복원
    for col in list_cols:
        df[col] = df[col].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
    
    encoded_parts = []
    
    # MLB 인코딩
    for col in list_cols:
        mlb = MultiLabelBinarizer()
        encoded = mlb.fit_transform(df[col])
        col_names = [f"{col}__{c}" for c in mlb.classes_]
        encoded_df = pd.DataFrame(encoded, columns=col_names, index=df.index)
        encoded_parts.append(encoded_df)
    
    # suncare만 spf_value 추가
    if cat == "suncare" and "spf_value" in df.columns:
        spf = df["spf_value"].astype(str).str.extract(r'(\d+)').astype(float)
        spf.columns = ["spf_value"]
        spf = spf.fillna(spf.median())
        encoded_parts.append(spf)
    
    X = pd.concat(encoded_parts, axis=1)
    y = df["y_final"]
    
    # 저장
    X.to_csv(path.replace("_cleaned.csv", "_X.csv"), index=False)
    y.to_csv(path.replace("_cleaned.csv", "_y.csv"), index=False)
    
    print(f"[{cat}] X: {X.shape}, y: {y.shape}")
    print(f"  피처: {list(X.columns)}\n")

[cleanser] X: (1212, 64), y: (1212,)
  피처: ['item_form_norm__balm', 'item_form_norm__capsule', 'item_form_norm__clay', 'item_form_norm__cleanser', 'item_form_norm__cream', 'item_form_norm__drops', 'item_form_norm__emulsion', 'item_form_norm__foam', 'item_form_norm__gel', 'item_form_norm__liquid', 'item_form_norm__lotion', 'item_form_norm__mask', 'item_form_norm__mitt', 'item_form_norm__mousse', 'item_form_norm__oil', 'item_form_norm__pad', 'item_form_norm__powder', 'item_form_norm__serum', 'item_form_norm__soap', 'item_form_norm__spray', 'item_form_norm__stick', 'item_form_norm__toner', 'item_form_norm__wipe', 'skin_type_norm__acne_prone', 'skin_type_norm__all', 'skin_type_norm__combination', 'skin_type_norm__dehydrated', 'skin_type_norm__dry', 'skin_type_norm__mature', 'skin_type_norm__normal', 'skin_type_norm__oily', 'skin_type_norm__sensitive', 'material_norm__alcohol_free', 'material_norm__aluminum_free', 'material_norm__cruelty_free', 'material_norm__dye_free', 'material_norm__fra

In [5]:
import pandas as pd
import ast

df = pd.read_csv(r"C:\workspace\finalproject\data\model_data_v1\model_cleanser_cleaned.csv")

list_cols = ["item_form_norm", "skin_type_norm", "material_norm", "benefit_norm"]

for col in list_cols:
    df[col] = df[col].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
    all_values = sorted(set(v for row in df[col] for v in row))
    print(f"\n[{col}] 총 {len(all_values)}개")
    for i, v in enumerate(all_values, 1):
        print(f"  {i:>3}. {v}")


[item_form_norm] 총 23개
    1. balm
    2. capsule
    3. clay
    4. cleanser
    5. cream
    6. drops
    7. emulsion
    8. foam
    9. gel
   10. liquid
   11. lotion
   12. mask
   13. mitt
   14. mousse
   15. oil
   16. pad
   17. powder
   18. serum
   19. soap
   20. spray
   21. stick
   22. toner
   23. wipe

[skin_type_norm] 총 9개
    1. acne_prone
    2. all
    3. combination
    4. dehydrated
    5. dry
    6. mature
    7. normal
    8. oily
    9. sensitive

[material_norm] 총 14개
    1. alcohol_free
    2. aluminum_free
    3. cruelty_free
    4. dye_free
    5. fragrance_free
    6. gluten_free
    7. mineral_oil_free
    8. oil_free
    9. paraben_free
   10. petroleum_free
   11. phthalate_free
   12. silicone_free
   13. sulfate_free
   14. vegan

[benefit_norm] 총 18개
    1. acne_control
    2. anti_aging
    3. brightening
    4. cleansing
    5. detoxifying
    6. even_toning
    7. exfoliating
    8. firming
    9. hydrating
   10. nourishing
   11. pore_treatme

In [6]:
import pandas as pd
import ast

files = {
    "skincare": r"C:\workspace\finalproject\data\model_data_v1\model_skincare_cleaned.csv",
    "mask":     r"C:\workspace\finalproject\data\model_data_v1\model_mask_cleaned.csv",
    "suncare":  r"C:\workspace\finalproject\data\model_data_v1\model_suncare_cleaned.csv",
}

list_cols = ["item_form_norm", "skin_type_norm", "material_norm", "benefit_norm"]

for cat, path in files.items():
    df = pd.read_csv(path)
    print(f"\n{'='*40}")
    print(f"[{cat.upper()}]")
    print(f"{'='*40}")
    
    for col in list_cols:
        df[col] = df[col].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
        all_values = sorted(set(v for row in df[col] for v in row))
        print(f"\n  [{col}] 총 {len(all_values)}개")
        for i, v in enumerate(all_values, 1):
            print(f"    {i:>3}. {v}")


[SKINCARE]

  [item_form_norm] 총 27개
      1. ampoule
      2. balm
      3. butter
      4. capsule
      5. clay
      6. cleanser
      7. cream
      8. drops
      9. emulsion
     10. essence
     11. foam
     12. gel
     13. hydrogel
     14. liquid
     15. lotion
     16. mask
     17. oil
     18. ointment
     19. pad
     20. patch
     21. powder
     22. serum
     23. sheet_mask
     24. spray
     25. stick
     26. toner
     27. wipe

  [skin_type_norm] 총 9개
      1. acne_prone
      2. all
      3. combination
      4. dehydrated
      5. dry
      6. mature
      7. normal
      8. oily
      9. sensitive

  [material_norm] 총 14개
      1. alcohol_free
      2. aluminum_free
      3. cruelty_free
      4. dye_free
      5. fragrance_free
      6. gluten_free
      7. mineral_oil_free
      8. oil_free
      9. paraben_free
     10. petroleum_free
     11. phthalate_free
     12. silicone_free
     13. sulfate_free
     14. vegan

  [benefit_norm] 총 25개
      1. ac

In [8]:
import pandas as pd
import ast

files = {
    "cleanser": r"C:\workspace\finalproject\data\model_data_v1\model_cleanser_cleaned.csv",
    "skincare": r"C:\workspace\finalproject\data\model_data_v1\model_skincare_cleaned.csv",
    "mask":     r"C:\workspace\finalproject\data\model_data_v1\model_mask_cleaned.csv",
    "suncare":  r"C:\workspace\finalproject\data\model_data_v1\model_suncare_cleaned.csv",
}

# item_form 제거 목록
item_form_remove = {"mitt", "wipe", "ointment"}

# benefit 합치기 매핑
benefit_merge = {
    "tightening":    "firming",
    "softening":     "smoothening",
    "whitening":     "brightening",
    "radiant_skin":  "brightening",
    "even_toning":   "brightening",
    "wrinkle_treatment": "anti_aging",
}

def clean_item_form(lst):
    return [v for v in lst if v not in item_form_remove]

def merge_benefits(lst):
    merged = [benefit_merge.get(v, v) for v in lst]
    return list(set(merged))  # 중복 제거

for cat, path in files.items():
    df = pd.read_csv(path)
    
    df["item_form_norm"] = df["item_form_norm"].apply(
        lambda x: clean_item_form(ast.literal_eval(x)) if isinstance(x, str) else x
    )
    df["benefit_norm"] = df["benefit_norm"].apply(
        lambda x: merge_benefits(ast.literal_eval(x)) if isinstance(x, str) else x
    )
    
    df.to_csv(path, index=False)
    print(f"[{cat}] 저장 완료")

    # 결과 확인
    all_forms = sorted(set(v for row in df["item_form_norm"] for v in row))
    all_benefits = sorted(set(v for row in df["benefit_norm"] for v in row))
    print(f"  item_form: {len(all_forms)}개 → {all_forms}")
    print(f"  benefit:   {len(all_benefits)}개 → {all_benefits}\n")

[cleanser] 저장 완료
  item_form: 21개 → ['balm', 'capsule', 'clay', 'cleanser', 'cream', 'drops', 'emulsion', 'foam', 'gel', 'liquid', 'lotion', 'mask', 'mousse', 'oil', 'pad', 'powder', 'serum', 'soap', 'spray', 'stick', 'toner']
  benefit:   13개 → ['acne_control', 'anti_aging', 'brightening', 'cleansing', 'detoxifying', 'exfoliating', 'firming', 'hydrating', 'nourishing', 'pore_treatment', 'rejuvenating', 'smoothening', 'soothing']

[skincare] 저장 완료
  item_form: 25개 → ['ampoule', 'balm', 'butter', 'capsule', 'clay', 'cleanser', 'cream', 'drops', 'emulsion', 'essence', 'foam', 'gel', 'hydrogel', 'liquid', 'lotion', 'mask', 'oil', 'pad', 'patch', 'powder', 'serum', 'sheet_mask', 'spray', 'stick', 'toner']
  benefit:   19개 → ['acne_control', 'anti_aging', 'antioxidant', 'brightening', 'cleansing', 'dark_circle', 'dark_spot', 'detoxifying', 'exfoliating', 'firming', 'hydrating', 'nourishing', 'oil_control', 'pore_treatment', 'redness_relief', 'rejuvenating', 'smoothening', 'soothing', 'uv_pr

In [9]:
import pandas as pd
import ast
from sklearn.preprocessing import MultiLabelBinarizer

files = {
    "cleanser": r"C:\workspace\finalproject\data\model_data_v1\model_cleanser_cleaned.csv",
    "skincare": r"C:\workspace\finalproject\data\model_data_v1\model_skincare_cleaned.csv",
    "mask":     r"C:\workspace\finalproject\data\model_data_v1\model_mask_cleaned.csv",
    "suncare":  r"C:\workspace\finalproject\data\model_data_v1\model_suncare_cleaned.csv",
}

list_cols = ["item_form_norm", "skin_type_norm", "material_norm", "benefit_norm"]

for cat, path in files.items():
    df = pd.read_csv(path)
    
    for col in list_cols:
        df[col] = df[col].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
    
    encoded_parts = []
    
    for col in list_cols:
        mlb = MultiLabelBinarizer()
        encoded = mlb.fit_transform(df[col])
        col_names = [f"{col}__{c}" for c in mlb.classes_]
        encoded_df = pd.DataFrame(encoded, columns=col_names, index=df.index)
        encoded_parts.append(encoded_df)
    
    if cat == "suncare" and "spf_value" in df.columns:
        spf = df["spf_value"].astype(str).str.extract(r'(\d+)').astype(float)
        spf.columns = ["spf_value"]
        spf = spf.fillna(spf.median())
        encoded_parts.append(spf)
    
    X = pd.concat(encoded_parts, axis=1)
    y = df["y_final"]
    
    X.to_csv(path.replace("_cleaned.csv", "_X.csv"), index=False)
    y.to_csv(path.replace("_cleaned.csv", "_y.csv"), index=False)
    
    print(f"[{cat}] X: {X.shape}, y: {y.shape}")
    

[cleanser] X: (1212, 57), y: (1212,)
[skincare] X: (4412, 67), y: (4412,)
[mask] X: (1226, 61), y: (1226,)
[suncare] X: (396, 40), y: (396,)


In [12]:
# other 비율 확인
for col in ["skin_type_clean", "material_clean", "benefits_clean"]:
    other_count = df[col].apply(lambda x: "other" in x).sum()
    print(f"{col} other 비율: {other_count}/{len(df)} ({other_count/len(df)*100:.1f}%)")

skin_type_clean other 비율: 0/396 (0.0%)
material_clean other 비율: 2/396 (0.5%)
benefits_clean other 비율: 81/396 (20.5%)


In [13]:
for cat, path in files.items():
    df = pd.read_csv(path)
    print(f"[{cat}] {len(df)}행")

[cleanser] 1234행
[skincare] 4534행
[mask] 1310행
[suncare] 396행


In [14]:
for cat, path in files.items():
    df = pd.read_csv(path)
    
    df["skin_type_clean"] = df["skin_type"].apply(lambda x: map_column(x, skin_type_map))
    df["material_clean"]  = df["material_type_free"].apply(lambda x: map_column(x, material_map))
    df["benefits_clean"]  = df["product_benefits"].apply(lambda x: map_column(x, benefit_map))
    
    before = len(df)
    
    if cat != "suncare":
        # other 행 삭제
        df = df[df["benefits_clean"].apply(lambda x: x != ["other"])]
    
    after = len(df)
    print(f"[{cat}] {before} → {after}행 ({before - after}개 삭제)")
    
    out_path = path.replace("_final.csv", "_cleaned.csv")
    df.to_csv(out_path, index=False)
    print(f"  저장 완료 → {out_path}")

[cleanser] 1234 → 1212행 (22개 삭제)
  저장 완료 → C:\workspace\finalproject\data\model_data_v1\model_cleanser_cleaned.csv
[skincare] 4534 → 4412행 (122개 삭제)
  저장 완료 → C:\workspace\finalproject\data\model_data_v1\model_skincare_cleaned.csv
[mask] 1310 → 1226행 (84개 삭제)
  저장 완료 → C:\workspace\finalproject\data\model_data_v1\model_mask_cleaned.csv
[suncare] 396 → 396행 (0개 삭제)
  저장 완료 → C:\workspace\finalproject\data\model_data_v1\model_suncare_cleaned.csv


In [15]:
import ast

df = pd.read_csv(r"C:\workspace\finalproject\data\model_data_v1\model_cleanser_cleaned.csv")

# 리스트 컬럼 복원
for col in ["skin_type_clean", "material_clean", "benefits_clean"]:
    df[col] = df[col].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

print(type(df["benefits_clean"].iloc[0]))  # <class 'list'> 나오면 OK
print(df["benefits_clean"].iloc[0])


<class 'list'>
['exfoliating']


In [10]:
# other로 분류된 원본값 확인
for cat, path in files.items():
    df = pd.read_csv(path)
    df["skin_type_clean"] = df["skin_type"].apply(lambda x: map_column(x, skin_type_map))
    df["benefits_clean"]  = df["product_benefits"].apply(lambda x: map_column(x, benefit_map))

    print(f"\n[{cat}] skin_type other 원본값:")
    other_st = df[df["skin_type_clean"].apply(lambda x: x == ["other"])]["skin_type"]
    print(other_st.value_counts().head(20).to_string())

    print(f"\n[{cat}] benefits other 원본값:")
    other_bn = df[df["benefits_clean"].apply(lambda x: x == ["other"])]["product_benefits"]
    print(other_bn.value_counts().head(20).to_string())
    print()


[cleanser] skin_type other 원본값:
skin_type
All                                                       292
Skin  with  large  pores  and  excess  oil  production      1
油性                                                          1
All Conditions- SLS FREE                                    1

[cleanser] benefits other 원본값:
product_benefits
Half body is for perfect for arms and legs, or touch-ups on the go.                                 7
Smoothening                                                                                         5
Hypoallergenic                                                                                      3
Skin Care                                                                                           1
Smoother                                                                                            1
Full body gives you a quick and even application from head to toe with no streaks, mess or fuss.    1
Aids in proper application and comfort while usi

In [2]:
 
# ─────────────────────────────────────────
# 설정: 카테고리만 바꿔서 돌리면 됨
# ─────────────────────────────────────────
CATEGORY = "cleanser"  # "skincare" / "maskpack" / "cleanser" / "suncare"
FILE_PATH = f"model_{CATEGORY}_final.csv"
 
# 카테고리별 피처 정의
FEATURES_BASE = ["item_form", "skin_type", "material_type_free", "product_benefits"]
FEATURES_SUNCARE = FEATURES_BASE + ["spf_value"]
FEATURES = FEATURES_SUNCARE if CATEGORY == "suncare" else FEATURES_BASE
 
TARGET = "y_final"  # PCA로 도출한 최종 y값 컬럼명

In [3]:
# ─────────────────────────────────────────
# 1. 데이터 로드
# ─────────────────────────────────────────
df = pd.read_csv(r"C:\workspace\finalproject\data\model_data_v1\model_cleanser_final.csv")
print(f"\n[{CATEGORY.upper()}] 로드 완료: {df.shape}")
print(df[FEATURES + [TARGET]].isnull().sum().rename("결측치 수"))


[CLEANSER] 로드 완료: (1234, 17)
item_form             0
skin_type             0
material_type_free    0
product_benefits      0
y_final               0
Name: 결측치 수, dtype: int64


## 피처 수정을 위한 데이터 분석

In [1]:
import pandas as pd

df = pd.read_csv(
    r"C:\workspace\finalproject\data\model_data_v1\model_cleanser_final.csv",
    on_bad_lines='skip'
)
print(df.columns.tolist())
print(df.shape)

c:\Users\asiae\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
c:\Users\asiae\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


['parent_asin', 'title', 'item_form', 'skin_type', 'material_type_free', 'product_benefits', 'average_rating', 'rating_number', 'y2', 'model', 'y1', 'y_final', 'title_clean', 'item_form_norm', 'skin_type_norm', 'material_norm', 'benefit_norm']
(1234, 17)


In [2]:
import pandas as pd

df = pd.read_json(
    r"C:\workspace\finalproject\data\meta_Beauty_and_Personal_Care.jsonl",
    lines=True,
    nrows=5  # 5줄만 읽기
)
print(df.columns.tolist())
print(df.head())

['main_category', 'title', 'average_rating', 'rating_number', 'features', 'description', 'price', 'images', 'videos', 'store', 'categories', 'details', 'parent_asin', 'bought_together']
  main_category                                              title  \
0    All Beauty  Shiyeen 10 Colors Hair Chalk for Girls Gift, K...   
1    All Beauty  Ebbfurln Bob Wig Human Hair, 13x4 HD Lace Fron...   
2    All Beauty  Makeup brush cleaner and dryer electronic spin...   
3    All Beauty  3 Inch Clipper Guards, Hair Clipper Guide Comb...   
4    All Beauty  Cathy Doll L-Glutathione Magic Cream SPF 50 Wh...   

   average_rating  rating_number  \
0             3.9             57   
1             4.1             60   
2             4.4              7   
3             4.0             68   
4             4.0             65   

                                            features  \
0  [🌼[MEET YOUR HAIR COLOR NEEDS] Bright color, h...   
1  [Frontal Wigs Human Hair Material: 100% unproc...   
2       

In [3]:
import pandas as pd
import numpy as np

files = {
    "cleanser": r"C:\workspace\finalproject\data\model_data_v1\model_cleanser_y.csv",
    "skincare": r"C:\workspace\finalproject\data\model_data_v1\model_skincare_y.csv",
    "mask":     r"C:\workspace\finalproject\data\model_data_v1\model_mask_y.csv",
    "suncare":  r"C:\workspace\finalproject\data\model_data_v1\model_suncare_y.csv",
}

for cat, path in files.items():
    y = pd.read_csv(path).squeeze()
    print(f"\n[{cat}]")
    print(f"  10%: {y.quantile(0.1):.4f}")
    print(f"  20%: {y.quantile(0.2):.4f}")
    print(f"  30%: {y.quantile(0.3):.4f}")
    print(f"  40%: {y.quantile(0.4):.4f}")
    print(f"  50%: {y.quantile(0.5):.4f}")
    print(f"  60%: {y.quantile(0.6):.4f}")
    print(f"  70%: {y.quantile(0.7):.4f}")
    print(f"  80%: {y.quantile(0.8):.4f}")
    print(f"  상위30% 기준값: {y.quantile(0.7):.4f} 이상 → {(y >= y.quantile(0.7)).sum()}개 ({(y >= y.quantile(0.7)).mean()*100:.1f}%)")
    print(f"  상위40% 기준값: {y.quantile(0.6):.4f} 이상 → {(y >= y.quantile(0.6)).sum()}개 ({(y >= y.quantile(0.6)).mean()*100:.1f}%)")
    print(f"  상위50% 기준값: {y.quantile(0.5):.4f} 이상 → {(y >= y.quantile(0.5)).sum()}개 ({(y >= y.quantile(0.5)).mean()*100:.1f}%)")


[cleanser]
  10%: 0.6318
  20%: 0.7275
  30%: 0.7657
  40%: 0.7960
  50%: 0.8186
  60%: 0.8370
  70%: 0.8513
  80%: 0.8679
  상위30% 기준값: 0.8513 이상 → 364개 (30.0%)
  상위40% 기준값: 0.8370 이상 → 485개 (40.0%)
  상위50% 기준값: 0.8186 이상 → 606개 (50.0%)

[skincare]
  10%: 0.5783
  20%: 0.6618
  30%: 0.7072
  40%: 0.7353
  50%: 0.7578
  60%: 0.7773
  70%: 0.7929
  80%: 0.8103
  상위30% 기준값: 0.7929 이상 → 1324개 (30.0%)
  상위40% 기준값: 0.7773 이상 → 1765개 (40.0%)
  상위50% 기준값: 0.7578 이상 → 2206개 (50.0%)

[mask]
  10%: 0.5732
  20%: 0.6757
  30%: 0.7306
  40%: 0.7634
  50%: 0.7865
  60%: 0.8044
  70%: 0.8225
  80%: 0.8451
  상위30% 기준값: 0.8225 이상 → 368개 (30.0%)
  상위40% 기준값: 0.8044 이상 → 491개 (40.0%)
  상위50% 기준값: 0.7865 이상 → 613개 (50.0%)

[suncare]
  10%: 0.4747
  20%: 0.5753
  30%: 0.6334
  40%: 0.6805
  50%: 0.7237
  60%: 0.7488
  70%: 0.7779
  80%: 0.8059
  상위30% 기준값: 0.7779 이상 → 119개 (30.1%)
  상위40% 기준값: 0.7488 이상 → 159개 (40.2%)
  상위50% 기준값: 0.7237 이상 → 198개 (50.0%)
